# Protein Builder v16 — preparo genérico pré-MD

Pipeline **agnóstico ao engine**: modela, repara e protona a estrutura e
devolve um PDB limpo, pronto para **GROMACS** (`pdb2gmx`) **ou OpenMM**
(`addSolvent`/`createSystem`). Não monta caixa, não adiciona água/íons, não
roda produção. O OpenMM aparece só como **minimização leve** de alívio de
choques.

**Etapas**
1. Extração de cadeia(s)
2. Sequência-alvo (opcional)
3. MODELLER AutoModel (opcional — reconstrói a cadeia inteira)
4. Reparo — PDBFixer (não-padrão, ausentes, heteroátomos seletivos) + **detecção de lacunas**
5. **Loop refinement** (opcional — refina só as lacunas, preserva o resto)
6. Protonação — PDB2PQR+PROPKA (`--ffout`) ou OpenMM `addHydrogens`
7. Minimização leve — OpenMM (implícito GBn2 ou vácuo)
8. Relatório de proveniência

> **Loop modeling:** o PDBFixer **insere** os resíduos ausentes (detectados
> por `findMissingResidues`) e o MODELLER (`DOPELoopModel`) **refina apenas
> esses loops** via `select_loop_atoms`. As regiões resolvidas ficam intactas,
> e o estágio só age quando há lacunas.

## 0. Imports

In [ ]:
import os, json, shutil, platform, subprocess, contextlib
from collections import defaultdict
from pathlib import Path

import requests
from Bio import SeqIO
from Bio.PDB import PDBParser, PDBIO, Select
from Bio.PDB.Polypeptide import is_aa
from Bio.SeqUtils import seq1

from pdbfixer import PDBFixer
import openmm
from openmm import LangevinMiddleIntegrator, Platform
from openmm.app import PDBFile, ForceField, Modeller, Simulation, NoCutoff, HBonds
from openmm.unit import kelvin, picosecond, femtoseconds, nanometer, kilojoules_per_mole

## 1. Configuração

`FF_FAMILY` controla só o esquema de nomes de resíduo da protonação e o FF da
minimização leve — não é o FF da produção.

In [ ]:
# ---- Entrada ----
INPUT_STRUCTURE = "1XNB.pdb"       # .pdb ou .cif
CHAIN_IDS       = ["A"]

# ---- Sequencia-alvo (so p/ MODELLER) ----
USE_FASTA   = True
FASTA_FILE  = "rcsb_pdb_1XNB.fasta"
USE_UNIPROT = False
UNIPROT_ID  = ""

# ---- Homology modeling (AutoModel: reconstroi a cadeia inteira) ----
USE_MODELLER        = False
NUM_MODELS          = 7
MIN_IDENTIDADE_ALVO = 0.7          # ~0.7 se template == estrutura; ~0.30 p/ homologo

# ---- Loop modeling (preserva estrutura; refina so lacunas internas) ----
LOOP_REFINE   = True              # so age se houver lacunas
LOOP_MODELS   = 7
LOOP_MD_LEVEL = "slow"             # "fast" | "slow"

# ---- Heteroatomos ----
KEEP_WATER   = False
KEEP_METALS  = False
KEEP_LIGANDS = False

# ---- Reparo ----
BUILD_MISSING_LOOPS   = True
DROP_TERMINAL_MISSING = True

# ---- Protonacao ----
PROTONATION_ENGINE = "pdb2pqr"     # "pdb2pqr" | "openmm"
USE_PROPKA         = True
PH                 = 5.7
SAVE_PROTONATION_SNAPSHOTS = True

# ---- Familia de FF ----
FF_FAMILY = "AMBER"                # AMBER | CHARMM

# ---- Minimizacao leve ----
MINIMIZE           = False
MIN_SOLVENT        = "implicit"    # "implicit" (GBn2, so AMBER) | "vacuum"
MIN_MAX_ITERATIONS = 1000
MIN_TOLERANCE_KJ   = 10.0
PRESERVE_FF_NAMES  = True          # mantem HID/HIE/HIP ou HSD/HSE/HSP no PDB minimizado
PLATFORM_NAME      = None

OUTPUT_DIR = f"output_{Path(INPUT_STRUCTURE).stem}_{''.join(CHAIN_IDS)}"
PROV = {"steps": []}

### Presets de campo de força

In [ ]:
FF_PRESETS = {
    "AMBER":  {"min_xml": ["amber14-all.xml"], "implicit_xml": "implicit/gbn2.xml",
               "pdb2pqr_ff": "AMBER",  "ffout": "AMBER"},
    "CHARMM": {"min_xml": ["charmm36.xml"],    "implicit_xml": None,
               "pdb2pqr_ff": "CHARMM", "ffout": "CHARMM"},
}

def resolve_forcefield():
    if FF_FAMILY == "OPLS":
        raise ValueError("OPLS nao e nativo em PDB2PQR/OpenMM(core). Use AMBER ou CHARMM.")
    if FF_FAMILY not in FF_PRESETS:
        raise ValueError(f"FF_FAMILY desconhecido: {FF_FAMILY}")
    preset = dict(FF_PRESETS[FF_FAMILY])
    solvent = MIN_SOLVENT
    if solvent == "implicit" and not preset["implicit_xml"]:
        print(f"Aviso: implicit indisponivel p/ {FF_FAMILY}; usando vacuo."); solvent = "vacuum"
    preset["solvent"] = solvent
    return preset

CFG = resolve_forcefield()
CFG

### Utilitários e diretórios

In [ ]:
_STD_AA = {"ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE","LEU","LYS",
           "MET","PHE","PRO","SER","THR","TRP","TYR","VAL","HID","HIE","HIP","HSD",
           "HSE","HSP","CYX","CYM","ASH","GLH","LYN","TYM"}
_METALS = {"ZN","MG","MN","FE","FE2","CA","NA","K","CU","CU1","NI","CO","CD","MO","W"}

def is_amino(name): return name.strip().upper() in _STD_AA
def sh(path, *parts): return os.path.join(path, *parts)

def step(msg):
    print("\n" + "=" * 72); print(msg); print("=" * 72)
    PROV["steps"].append(msg)

class ChainSelect(Select):
    def __init__(self, chain_ids): self.chain_ids = set(chain_ids)
    def accept_chain(self, chain): return chain.id in self.chain_ids

def load_structure(path):
    ext = Path(path).suffix.lower()
    if ext in (".cif", ".mmcif"):
        from Bio.PDB import MMCIFParser
        return MMCIFParser(QUIET=True).get_structure("input", path)
    return PDBParser(QUIET=True).get_structure("input", path)

for folder in ["raw","modelled","fixed","loops","protonated","minimized","pqr","logs","report"]:
    os.makedirs(sh(OUTPUT_DIR, folder), exist_ok=True)
print("Diretorio:", OUTPUT_DIR)

## 2. Extração de cadeia(s)

In [ ]:
def extract_chains():
    step("[1/8] Extraindo cadeia(s)")
    structure = load_structure(INPUT_STRUCTURE)
    chains = [c.id for c in structure.get_chains()]
    print("Cadeias encontradas:", ", ".join(chains))
    faltando = [c for c in CHAIN_IDS if c not in chains]
    if faltando:
        raise ValueError(f"Cadeia(s) {faltando} ausente(s). Disponiveis: {chains}")
    out = sh(OUTPUT_DIR, "raw", f"chain_{''.join(CHAIN_IDS)}.pdb")
    io = PDBIO(); io.set_structure(structure); io.save(out, ChainSelect(CHAIN_IDS))
    print("OK ->", out); return out

chain = extract_chains()

## 3. Sequência-alvo + checagem (só p/ MODELLER)

A checagem usa **`Bio.Align.PairwiseAligner`** (identidade real de alinhamento
global), no lugar do `difflib`. Reporta a métrica que você de fato usaria em
modelagem comparativa; calibre `MIN_IDENTIDADE_ALVO` ao tipo de modelagem.

In [ ]:
def load_sequence():
    step("[2/8] Carregando sequencia-alvo")
    if USE_FASTA and Path(FASTA_FILE).exists():
        seq = str(next(SeqIO.parse(FASTA_FILE, "fasta")).seq).upper()
        print(f"OK: FASTA ({len(seq)} aa)"); return seq
    if USE_UNIPROT and UNIPROT_ID:
        r = requests.get(f"https://rest.uniprot.org/uniprotkb/{UNIPROT_ID}.fasta", timeout=30)
        r.raise_for_status()
        seq = "".join(r.text.splitlines()[1:]).upper()
        print(f"OK: UniProt {UNIPROT_ID} ({len(seq)} aa)"); return seq
    print("Aviso: sem FASTA/UniProt."); return None

def extract_existing_sequence(chain_pdb):
    letras = []
    for res in load_structure(chain_pdb).get_residues():
        if is_aa(res, standard=False):
            aa = seq1(res.get_resname().strip().upper())
            if aa != "X": letras.append(aa)
    return "".join(letras)

def _identidade_pairwise(a, b):
    from Bio.Align import PairwiseAligner
    aligner = PairwiseAligner()
    aligner.mode = "global"
    aligner.match_score = 1.0; aligner.mismatch_score = 0.0
    aligner.open_gap_score = -1.0; aligner.extend_gap_score = -0.5
    aln = aligner.align(a, b)[0]
    ident = 0
    for (a0, a1), (b0, b1) in zip(*aln.aligned):
        ident += sum(x == y for x, y in zip(a[a0:a1], b[b0:b1]))
    return ident / max(1, min(len(a), len(b)))

def checar_compatibilidade_sequencia(chain_pdb, target):
    est = extract_existing_sequence(chain_pdb)
    if not est or not target: return
    ident = _identidade_pairwise(est, target)
    print(f"Identidade (alinhamento global): {ident:.1%}")
    PROV["identidade_alvo"] = ident
    if ident < MIN_IDENTIDADE_ALVO:
        raise ValueError(f"So {ident:.1%} de identidade: confira UNIPROT_ID/FASTA_FILE "
                         "(ou baixe MIN_IDENTIDADE_ALVO se for modelagem comparativa).")

seq = load_sequence()

## 4. MODELLER AutoModel (opcional)

`AutoModel` reconstrói a **cadeia inteira** — descarta coordenadas
experimentais. Para lacunas, prefira o **loop refinement** (seção 6), que
preserva a estrutura. Padrão OFF.

In [ ]:
def modeller_strong(chain_pdb, target):
    if not USE_MODELLER or target is None:
        return chain_pdb, None
    step("[3/8] MODELLER (AutoModel)")
    print("AVISO: AutoModel reconstroi a cadeia inteira; p/ lacunas use LOOP_REFINE.")
    checar_compatibilidade_sequencia(chain_pdb, target)
    from modeller import Environ, Model, Alignment
    from modeller.automodel import AutoModel
    env = Environ(); env.io.atom_files_directory = [".", os.path.dirname(chain_pdb)]
    code_ = Path(chain_pdb).stem
    aln = Alignment(env); aln.append_model(Model(env, file=code_), align_codes=code_)
    tgt = sh(OUTPUT_DIR, "logs", "target.seq")
    with open(tgt, "w") as f:
        f.write(">P1;target\nsequence:target::::::::\n" + target + "*\n")
    aln.append(file=tgt, align_codes="target"); aln.align2d()
    ali = sh(OUTPUT_DIR, "logs", "alignment.ali"); aln.write(file=ali, alignment_format="PIR")
    with open(sh(OUTPUT_DIR, "logs", "modeller.log"), "w") as log:
        with contextlib.redirect_stdout(log):
            am = AutoModel(env, alnfile=ali, knowns=code_, sequence="target")
            am.starting_model = 1; am.ending_model = NUM_MODELS; am.make()
    ranking = [{"model": o["name"], "molpdf": o.get("molpdf"),
                "dope_score": o.get("DOPE score"), "ga341": o.get("GA341 score")}
               for o in am.outputs if o.get("failure") is None]
    if not ranking:
        print("Aviso: MODELLER falhou."); return chain_pdb, None
    ranking.sort(key=lambda d: (d["dope_score"] is None, d["dope_score"]))
    best = ranking[0]
    out = sh(OUTPUT_DIR, "modelled", "best_model.pdb"); shutil.copy(best["model"], out)
    print("OK ->", best["model"]); return out, best

model, best_model_info = modeller_strong(chain, seq)
model

## 5. Reparo — PDBFixer + detecção de lacunas

Converte não-padrão (MSE→MET), reconstrói ausentes (opção de descartar caudas
terminais) e remove heteroátomos seletivamente. Também **detecta os resíduos
inseridos** (diferença presente-depois − presente-antes) e os agrupa em
intervalos contíguos — é o que alimenta o loop refinement.

In [ ]:
def _present_residues(topology):
    present = set()
    for chain in topology.chains():
        for res in chain.residues():
            try: present.add((chain.id, int(res.id)))
            except (TypeError, ValueError): pass
    return present

def _contiguous_ranges(residues):
    bychain = defaultdict(list)
    for c, n in residues: bychain[c].append(n)
    ranges = []
    for c, nums in bychain.items():
        nums = sorted(set(nums)); start = prev = nums[0]
        for n in nums[1:]:
            if n == prev + 1: prev = n
            else: ranges.append((c, start, prev)); start = prev = n
        ranges.append((c, start, prev))
    return ranges

def _handle_heterogens(fixer):
    if KEEP_LIGANDS:
        print("KEEP_LIGANDS=True -> cofatores preservados (params no engine).")
        if not KEEP_WATER: fixer.removeHeterogens(keepWater=False)
        return
    m = Modeller(fixer.topology, fixer.positions); to_del = []
    for res in m.topology.residues():
        name = res.name.strip().upper()
        if is_amino(name): continue
        if name in ("HOH","WAT","TIP3","SPC","H2O") and KEEP_WATER: continue
        if name in _METALS and KEEP_METALS: continue
        to_del.append(res)
    if to_del: m.delete(to_del)
    fixer.topology, fixer.positions = m.topology, m.positions

def pdbfixer_stage(pdb_file):
    step("[4/8] PDBFixer (reparo)")
    fixer = PDBFixer(filename=pdb_file)
    fixer.findNonstandardResidues()
    if fixer.nonstandardResidues:
        print("Nao-padrao:", [(r.name, new) for r, new in fixer.nonstandardResidues])
        fixer.replaceNonstandardResidues()
    fixer.findMissingResidues()
    if not BUILD_MISSING_LOOPS:
        fixer.missingResidues = {}
    elif DROP_TERMINAL_MISSING:
        chains = list(fixer.topology.chains())
        for key in list(fixer.missingResidues.keys()):
            ci, ri = key
            if ri == 0 or ri == len(list(chains[ci].residues())):
                del fixer.missingResidues[key]
    present_before = _present_residues(fixer.topology)
    fixer.findMissingAtoms(); fixer.addMissingAtoms()
    inserted = _present_residues(fixer.topology) - present_before
    loop_ranges = _contiguous_ranges(inserted) if inserted else []
    if loop_ranges:
        print("Lacunas internas inseridas:", [f"{c}:{s}-{e}" for c,s,e in loop_ranges])
        PROV["loop_residues"] = [f"{c}:{s}-{e}" for c,s,e in loop_ranges]
    _handle_heterogens(fixer)
    out = sh(OUTPUT_DIR, "fixed", "fixed.pdb")
    with open(out, "w") as f:
        PDBFile.writeFile(fixer.topology, fixer.positions, f, keepIds=True)
    print("OK ->", out); return out, loop_ranges

fixed, loop_ranges = pdbfixer_stage(model)
fixed, loop_ranges

## 6. Loop refinement (opcional)

Refina **apenas** as lacunas detectadas na seção 5, via `select_loop_atoms`.
Toda a estrutura resolvida fica fixa. Só age se `LOOP_REFINE=True` **e** houver
lacunas — do contrário devolve o `fixed.pdb` inalterado.

In [ ]:
def loop_refine_stage(fixed_pdb, loop_ranges):
    if not LOOP_REFINE:
        return fixed_pdb
    if not loop_ranges:
        step("[5/8] Loop refinement (sem lacunas -> nada a refinar)")
        return fixed_pdb
    step("[5/8] Loop refinement (MODELLER DOPELoopModel)")
    print("Refinando:", [f"{c}:{s}-{e}" for c,s,e in loop_ranges])
    from modeller import Environ, Selection
    from modeller.automodel import DOPELoopModel, refine, assess
    work_dir = os.path.dirname(os.path.abspath(fixed_pdb))
    inimodel = Path(fixed_pdb).name
    md_level = refine.slow if LOOP_MD_LEVEL == "slow" else refine.fast

    class _AutoLoop(DOPELoopModel):
        def select_loop_atoms(self):
            segs = [self.residue_range(f"{s}:{c}", f"{e}:{c}") for (c,s,e) in loop_ranges]
            return Selection(*segs)

    env = Environ(); env.io.atom_files_directory = [".", work_dir]
    with open(sh(OUTPUT_DIR, "logs", "loopmodel.log"), "w") as log:
        with contextlib.redirect_stdout(log):
            m = _AutoLoop(env, inimodel=inimodel, sequence="loops",
                          loop_assess_methods=(assess.DOPE,))
            m.loop.starting_model = 1; m.loop.ending_model = LOOP_MODELS
            m.loop.md_level = md_level; m.make()
    outs = [o for o in m.loop.outputs if o.get("failure") is None]
    if not outs:
        print("Aviso: loop refinement falhou; usando PDBFixer."); return fixed_pdb
    outs.sort(key=lambda o: (o.get("DOPE score") is None, o.get("DOPE score")))
    best = outs[0]
    out = sh(OUTPUT_DIR, "loops", "loops_refined.pdb"); shutil.copy(best["name"], out)
    PROV["loop_refinement"] = {"models": LOOP_MODELS, "md_level": LOOP_MD_LEVEL,
                               "best": best["name"], "dope": best.get("DOPE score")}
    print("OK ->", best["name"]); return out

refined = loop_refine_stage(fixed, loop_ranges)
refined

## 7. Protonação

`pdb2pqr` (PROPKA) com **`--ffout`** grava os estados nos nomes de resíduo; ou
`openmm` via `Modeller.addHydrogens(pH=…)`.

In [ ]:
def _protonate_pdb2pqr(pdb_file):
    out_pdb = sh(OUTPUT_DIR, "protonated", "protonated.pdb")
    out_pqr = sh(OUTPUT_DIR, "pqr", "protein.pqr")
    if SAVE_PROTONATION_SNAPSHOTS:
        shutil.copy(pdb_file, sh(OUTPUT_DIR, "report", "before_protonation.pdb"))
    cmd = ["pdb2pqr30", f"--ff={CFG['pdb2pqr_ff']}", f"--ffout={CFG['ffout']}",
           f"--with-ph={PH}", "--keep-chain"]
    if USE_PROPKA: cmd += ["--titration-state-method", "propka"]
    cmd += ["--pdb-output", out_pdb, pdb_file, out_pqr]
    print("Rodando:", " ".join(cmd)); subprocess.run(cmd, check=True)
    if SAVE_PROTONATION_SNAPSHOTS:
        shutil.copy(out_pdb, sh(OUTPUT_DIR, "report", "after_protonation.pdb"))
    PROV["protonation"] = {"engine": "pdb2pqr+propka", "ph": PH, "ffout": CFG["ffout"]}
    print("OK ->", out_pdb); return out_pdb

def _protonate_openmm(pdb_file):
    pdb = PDBFile(pdb_file); ff = ForceField(*CFG["min_xml"])
    m = Modeller(pdb.topology, pdb.positions); m.addHydrogens(ff, pH=PH)
    out_pdb = sh(OUTPUT_DIR, "protonated", "protonated.pdb")
    with open(out_pdb, "w") as f:
        PDBFile.writeFile(m.topology, m.positions, f, keepIds=True)
    PROV["protonation"] = {"engine": "openmm.addHydrogens", "ph": PH}
    print("OK ->", out_pdb); return out_pdb

def protonate(pdb_file):
    step("[6/8] Protonacao")
    return _protonate_openmm(pdb_file) if PROTONATION_ENGINE == "openmm" else _protonate_pdb2pqr(pdb_file)

protonated = protonate(refined)
protonated

### QC da protonação

In [ ]:
def compare_protonation():
    if not SAVE_PROTONATION_SNAPSHOTS: return []
    before = sh(OUTPUT_DIR, "report", "before_protonation.pdb")
    after  = sh(OUTPUT_DIR, "report", "after_protonation.pdb")
    if not (os.path.exists(before) and os.path.exists(after)): return []
    p = PDBParser(QUIET=True)
    def keymap(s): return {(r.get_parent().id, r.id[1], r.id[2]): r for r in s.get_residues()}
    bm, am = keymap(p.get_structure("b", before)), keymap(p.get_structure("a", after))
    titr = {"HIS","ASP","GLU","CYS","LYS","TYR"}; linhas = []
    for key in sorted(set(bm) & set(am)):
        rb, ra = bm[key], am[key]
        if rb.get_resname() in titr and rb.get_resname() != ra.get_resname():
            ch, num, ic = key
            linhas.append(f"Cadeia {ch} res {num}{ic.strip()}: {rb.get_resname()} -> {ra.get_resname()}")
    with open(sh(OUTPUT_DIR, "report", "protonation_changes.txt"), "w") as f:
        f.write("\n".join(linhas) if linhas else "Nenhuma mudanca detectada.")
    PROV["protonation_changes"] = linhas
    print(f"QC: {len(linhas)} residuo(s) com estado alterado"); return linhas

_ = compare_protonation()

## 8. Minimização leve (único uso do OpenMM)

Alívio de choques; solvente implícito (GBn2) por padrão. A minimização em
caixa acontece depois, no engine de produção.

In [ ]:
def _write_coords_preserving_names(template_pdb, positions, out_path):
    from openmm.unit import angstrom
    coords = [p.value_in_unit(angstrom) for p in positions]
    out_lines, i, ok = [], 0, True
    with open(template_pdb) as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                if i >= len(coords) or len(line) < 54: ok = False; break
                x, y, z = coords[i]
                line = f"{line[:30]}{x:8.3f}{y:8.3f}{z:8.3f}{line[54:]}"; i += 1
            out_lines.append(line)
    if not ok or i != len(coords): return False
    with open(out_path, "w") as f: f.writelines(out_lines)
    return True

def _make_platform():
    if PLATFORM_NAME: return Platform.getPlatformByName(PLATFORM_NAME), {}
    for name in ("CUDA", "OpenCL", "CPU"):
        try:
            p = Platform.getPlatformByName(name)
            return p, ({"Precision": "mixed"} if name in ("CUDA","OpenCL") else {})
        except Exception: continue
    return None, {}

def minimize(pdb_file):
    if not MINIMIZE:
        step("[7/8] Minimizacao pulada"); return pdb_file
    step("[7/8] Minimizacao leve (OpenMM)")
    xmls = list(CFG["min_xml"])
    if CFG["solvent"] == "implicit" and CFG["implicit_xml"]: xmls.append(CFG["implicit_xml"])
    print("FF:", xmls, "| solvente:", CFG["solvent"])
    pdb = PDBFile(pdb_file); ff = ForceField(*xmls)
    system = ff.createSystem(pdb.topology, nonbondedMethod=NoCutoff, constraints=HBonds)
    integ = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 2*femtoseconds)
    plat, props = _make_platform()
    sim = Simulation(pdb.topology, system, integ, plat, props) if plat else \
          Simulation(pdb.topology, system, integ)
    if plat: print("Plataforma:", plat.getName())
    sim.context.setPositions(pdb.positions)
    e0 = sim.context.getState(getEnergy=True).getPotentialEnergy()
    sim.minimizeEnergy(tolerance=MIN_TOLERANCE_KJ*kilojoules_per_mole/nanometer,
                       maxIterations=MIN_MAX_ITERATIONS)
    st = sim.context.getState(getPositions=True, getEnergy=True); e1 = st.getPotentialEnergy()
    out = sh(OUTPUT_DIR, "minimized", "minimized.pdb")
    pos = st.getPositions(); preserved = False
    if PRESERVE_FF_NAMES:
        preserved = _write_coords_preserving_names(pdb_file, pos, out)
        print("Nomes de FF preservados." if preserved else
              "Aviso: contagem nao casou; nomes canonicos do OpenMM.")
    if not preserved:
        with open(out, "w") as f: PDBFile.writeFile(pdb.topology, pos, f, keepIds=True)
    PROV["minimization"] = {"solvent": CFG["solvent"], "energy_before": str(e0),
                            "energy_after": str(e1), "max_iterations": MIN_MAX_ITERATIONS,
                            "ff_names_preserved": preserved}
    print(f"OK: {e0} -> {e1}\n->", out); return out

final_file = minimize(protonated)
final_file

## Relatório de proveniência

In [ ]:
def write_report(best_model_info, final_file):
    versions = {"python": platform.python_version(), "openmm": openmm.version.version}
    try:
        import pdbfixer as _pf; versions["pdbfixer"] = getattr(_pf, "__version__", "?")
    except Exception: pass
    report = {
        "input": INPUT_STRUCTURE, "chains": CHAIN_IDS, "ff_family": FF_FAMILY,
        "residue_naming": CFG["ffout"], "pH": PH, "protonation_engine": PROTONATION_ENGINE,
        "loop_refine": LOOP_REFINE,
        "minimization_solvent": CFG["solvent"] if MINIMIZE else None,
        "keep": {"water": KEEP_WATER, "metals": KEEP_METALS, "ligands": KEEP_LIGANDS},
        "best_model": best_model_info, "final_file": final_file,
        "handoff": {"gromacs": "gmx pdb2gmx -f <final> (preserve os H)",
                    "openmm": "PDBFile(<final>) -> Modeller.addSolvent -> createSystem"},
        "versions": versions, "provenance": PROV,
    }
    path = sh(OUTPUT_DIR, "report", "preparation_report.json")
    with open(path, "w") as f: json.dump(report, f, indent=2, default=str)
    print("Relatorio ->", path); return report

report = write_report(best_model_info, final_file)
report

## Handoff para o engine de MD

- **GROMACS:** `gmx pdb2gmx -v -f protonated.pdb -o protein.gro -ignh`
  (preservando os H), depois `editconf`/`solvate`/`genion`.
- **OpenMM:** `PDBFile(minimized.pdb)` → `Modeller.addSolvent(ff, model=…,
  padding=…, ionicStrength=0.15*molar)` → `createSystem(nonbondedMethod=PME)`.

> **Cofatores/substratos** (KEEP_LIGANDS=True): parametrize no engine
> (openmmforcefields/GAFF2/OpenFF; para heme, use parâmetros publicados).